## Camada Silver - Limpeza e Transformações

In [0]:
from pyspark.sql import functions as F

### Uso do schema

In [0]:
catalog = "mvp_data_engeneering"
schema = "silver"
schema_origem = "bronze"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

## Análise de Qualidade de Dados e Transformação

Antes de modelarmos os dados para a camada Gold, é fundamental garantir a fiabilidade da informação (Etapa 4.5 do MVP). Analisaremos a **Completude, Unicidade, Consistência e Acurácia** das tabelas principais.

In [0]:
# Carregar tabelas brutas para análise
df_orders = spark.table(f"{catalog}.{schema_origem}.orders")
df_items = spark.table(f"{catalog}.{schema_origem}.order_items")

### 1. Verificação de Qualidade de Dados

**Completude (Valores Nulos):**
Vamos verificar se existem pedidos sem data de entrega (`order_delivered_customer_date`), o que pode impactar o nosso cálculo de SLA.

In [0]:
# Contar valores nulos nas datas de entrega
df_orders.select(
    F.sum(F.col("order_delivered_customer_date").isNull().cast("int")).alias("pedidos_sem_data_entrega"),
    F.count("*").alias("total_pedidos")
).display()

*Nota de Qualidade:* Descobrimos que existem registos sem data de entrega. Cruzando com a coluna `order_status`, percebemos que se tratam de pedidos "cancelados" ou "inativos". Como o nosso objetivo é analisar o fluxo logístico real, **decidimos filtrar a base apenas para pedidos com o status 'delivered'**.

**Unicidade e Consistência:**
Vamos confirmar se a chave primária `order_id` é única na tabela de pedidos e se existem anomalias (ex: fretes com valor negativo).

In [0]:
# Teste de Unicidade: Total de linhas vs Total de IDs distintos
df_orders.select(
    F.count("*").alias("total_linhas"),
    F.countDistinct("order_id").alias("ids_unicos")
).display()

# Teste de Acurácia/Outliers: Verificar se há valores de frete negativos ou iguais a zero
df_items.filter(F.col("freight_value").cast("double") < 0).select(
    F.count("*").alias("itens_com_frete_negativo")
).display()

df_items.filter(F.col("freight_value").cast("double") == 0).select(
    F.count("*").alias("itens_com_frete_zero")
).display()

*Nota de Qualidade:* A chave primária `order_id` validou perfeitamente (100% de unicidade). Na tabela de itens, não detectamos a presença de fretes com valor negativo, mas detectamos 383 instâncias de frete zero. Estes não são erros de acurácia, mas sim promoções legítimas de "Frete Grátis", logo, **não serão removidos**, mas é um *insight* importante.

### 2. Limpeza, Tipagem e Persistência na Silver

Com base na nossa auditoria de qualidade, aplicaremos as transformações: conversão de tipos (strings para timestamps), cálculo das *features* temporais (dias logísticos) e a remoção dos registos cancelados.

In [0]:
# 1. Transformação da Tabela Orders
df_orders_silver = df_orders \
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp")) \
    .withColumn("order_approved_at", F.to_timestamp("order_approved_at")) \
    .withColumn("order_delivered_carrier_date", F.to_timestamp("order_delivered_carrier_date")) \
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date")) \
    .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date")) \
    .withColumn("dias_processamento", F.datediff("order_delivered_carrier_date", "order_approved_at")) \
    .withColumn("dias_transporte", F.datediff("order_delivered_customer_date", "order_delivered_carrier_date")) \
    .withColumn("dias_atraso", F.datediff("order_delivered_customer_date", "order_estimated_delivery_date")) \
    .withColumn("is_atrasado", F.when(F.col("dias_atraso") > 0, 1).otherwise(0))

# Tratamento de Qualidade: Filtrar apenas pedidos entregues com sucesso
df_orders_silver = df_orders_silver.filter(F.col("order_status") == "delivered")
df_orders_silver.write.format("delta").mode("overwrite").saveAsTable("orders_cleaned")

# 2. Transformação da Tabela Reviews (Remoção de duplicatas)
df_reviews = spark.table(f"{catalog}.{schema_origem}.order_reviews")
df_reviews_silver = df_reviews \
    .withColumn("review_creation_date", F.to_timestamp("review_creation_date")) \
    .withColumn("review_score", F.col("review_score").cast("integer")) \
    .dropDuplicates(["order_id"]) # Tratamento de Unicidade: Garantir apenas 1 avaliação por pedido

df_reviews_silver.write.format("delta").mode("overwrite").saveAsTable("reviews_cleaned")

# 3. Transformação da Tabela de Itens e Vendedores
df_items_silver = df_items.select("order_id", "seller_id").dropDuplicates()
df_items_silver.write.format("delta").mode("overwrite").saveAsTable("order_sellers")

# 4. Cópias Diretas (Clientes e Vendedores)
spark.table(f"{catalog}.{schema_origem}.customers").write.format("delta").mode("overwrite").saveAsTable("customers_cleaned")
spark.table(f"{catalog}.{schema_origem}.sellers").write.format("delta").mode("overwrite").saveAsTable("sellers_cleaned")

print("Auditoria de Qualidade concluída. Tabelas limpas guardadas na Silver com sucesso.")